# MVA v4 · NB04D — targeted nephrocalcinosis secondary-explanation screen
**v1.0.0 · private variant table stays private · preregistered phenotype-mismatch audit**

## Question

The BUB1B pair explains the core MVA hypothesis, but nephrocalcinosis is not a characteristic defining feature
of classic BUB1B-MVA1. This notebook asks whether the already-annotated genome contains a **second plausible
genetic explanation for nephrocalcinosis/nephrolithiasis**.

This is not a search for a better MVA diagnosis and does not alter the Track-1 BUB1B ranking.

## Privacy

Input is the existing local/private VEP-derived Parquet:
`02_RESULTS_PRIVATE/NB01B_COMPOUND_HET_v1_0_0/variant_scored_private.parquet`

The notebook does **not** reopen or upload the raw VCF.

Full variant/gene results stay in `02_RESULTS_PRIVATE`.
The public handoff contains only counts/status, never secondary gene names, coordinates, or genotypes.

## Frozen primary gene panel

Primary panel is the 30-gene NCBI GTR nephrolithiasis/nephrocalcinosis panel frozen before reading patient rows:

ADCY10, AGXT, APRT, ATP6V0A4, ATP6V1B1, CA2, CASR, CLCN5, CLDN16, CLDN19, CYP24A1, FAM20A,
GRHPR, HNF4A, HOGA1, HPRT1, KCNJ1, OCRL, SLC12A1, SLC22A12, SLC26A1, SLC2A9, SLC34A1,
SLC34A3, SLC3A1, SLC4A1, SLC7A9, SLC9A3R1, VDR, XDH.

A signed-off Genomics England PanelApp v6.2 expansion may be used only as a **separate sensitivity layer** if
the exact version can be fetched. It cannot replace the frozen primary panel.

## Variant rules

Technical:
- FILTER PASS / empty acceptable;
- GQ >=20 when present;
- DP >=10 when present.

Biological:
- HIGH/MODERATE consequence or splice-region consequence;
- population max AF missing or <=0.01;
- for a monoallelic candidate: AF <=0.001 unless ClinVar is clean Pathogenic/Likely pathogenic;
- ClinVar strings with conflicts are not counted as P/LP by substring matching.

Gene-level priority:
- homozygous/hemizygous plausible hit;
- >=2 rare heterozygous hits in a recessive gene pattern (phase may remain unknown);
- one high-impact/known P-LP hit is retained for manual MOI review.

Output label is `finding_type=secondary`.
This is research triage, not a diagnosis.

In [ ]:
import os
from pathlib import Path
from datetime import datetime, timezone
import json, re, numpy as np, pandas as pd

from google.colab import drive
drive.mount("/content/drive",force_remount=False)

ROOT = Path(os.environ.get("MVA_ROOT", "/path/to/MVA_Hackathon_2026"))
PRIVATE_IN=ROOT/"02_RESULTS_PRIVATE"/"NB01B_COMPOUND_HET_v1_0_0"/"variant_scored_private.parquet"
PRIVATE_OUT=ROOT/"02_RESULTS_PRIVATE"/"NB04D_NEPHROCALCINOSIS_SECONDARY_v1_0_0"
PUBLIC_OUT=ROOT/"03_PUBLIC_ARTIFACTS"
PRIVATE_OUT.mkdir(parents=True,exist_ok=True); PUBLIC_OUT.mkdir(parents=True,exist_ok=True)

CORE30=['ADCY10', 'AGXT', 'APRT', 'ATP6V0A4', 'ATP6V1B1', 'CA2', 'CASR', 'CLCN5', 'CLDN16', 'CLDN19', 'CYP24A1', 'FAM20A', 'GRHPR', 'HNF4A', 'HOGA1', 'HPRT1', 'KCNJ1', 'OCRL', 'SLC12A1', 'SLC22A12', 'SLC26A1', 'SLC2A9', 'SLC34A1', 'SLC34A3', 'SLC3A1', 'SLC4A1', 'SLC7A9', 'SLC9A3R1', 'VDR', 'XDH']
assert len(CORE30)==30
assert PRIVATE_IN.exists(), f"Missing existing private annotated table: {PRIVATE_IN}"
print("Frozen primary panel:",len(CORE30),"genes")

## 1. Load only the existing private annotated table

In [ ]:
v=pd.read_parquet(PRIVATE_IN)
print("Rows loaded privately:",len(v))
print("Columns:",list(v.columns))

required={"chrom","pos","ref","alt","gene","consequence","impact","zygosity"}
missing=required-set(v.columns)
assert not missing, f"Required columns missing: {missing}"

## 2. Exact technical/population/consequence filters

In [ ]:
def clean_plp(s):
    s=str(s or "").lower().replace("_"," ")
    # Explicitly exclude conflicts / uncertain / benign mixtures.
    if any(x in s for x in ["conflict","uncertain","benign"]):
        return False
    toks=re.split(r"[,&;|/]+",s)
    return any(t.strip() in {"pathogenic","likely pathogenic","pathogenic likely pathogenic"} for t in toks)

def numeric_col(df,name,default=np.nan):
    return pd.to_numeric(df[name],errors="coerce") if name in df.columns else pd.Series(default,index=df.index)

x=v[v.gene.astype(str).isin(CORE30)].copy()
x["gq_num"]=numeric_col(x,"gq")
x["dp_num"]=numeric_col(x,"dp")
x["max_af_num"]=numeric_col(x,"max_af")
x["clean_clinvar_plp"]=x["clin_sig"].map(clean_plp) if "clin_sig" in x.columns else False

filter_ok = x["filter"].astype(str).isin(["PASS",".","","None","nan"]) if "filter" in x.columns else pd.Series(True,index=x.index)
gq_ok = x.gq_num.isna() | (x.gq_num>=20)
dp_ok = x.dp_num.isna() | (x.dp_num>=10)

cons=x.consequence.fillna("").astype(str)
impact=x.impact.fillna("").astype(str).str.upper()
consequence_ok=impact.isin(["HIGH","MODERATE"]) | cons.str.contains(
    r"splice_(?:donor|acceptor|region)|transcript_ablation|stop_gained|frameshift|start_lost",
    case=False,regex=True
)
af_ok=x.max_af_num.isna() | (x.max_af_num<=0.01)

hits=x[filter_ok & gq_ok & dp_ok & consequence_ok & af_ok].copy()
hits["ultrarare_or_plp"]=hits.max_af_num.isna() | (hits.max_af_num<=0.001) | hits.clean_clinvar_plp

print("Panel rows before filters:",len(x))
print("Variant hits after frozen filters:",len(hits))

## 3. Gene-level secondary-finding triage

In [ ]:
def is_hom(z):
    s=str(z).lower()
    return any(k in s for k in ["hom_alt","homozyg","1/1","1|1","hemiz"])

gene_rows=[]
for gene,g in hits.groupby("gene"):
    rare=g[g.ultrarare_or_plp].copy()
    n_rare=len(rare)
    n_het=int(rare.zygosity.astype(str).str.lower().str.contains("het|0/1|1/0|0\\|1|1\\|0",regex=True).sum())
    n_hom=int(rare.zygosity.map(is_hom).sum())
    n_high=int((rare.impact.astype(str).str.upper()=="HIGH").sum())
    n_plp=int(rare.clean_clinvar_plp.sum())

    if n_hom>=1:
        status="HIGH_REVIEW"
        reason="homozygous/hemizygous rare plausible hit"
    elif n_het>=2:
        status="HIGH_REVIEW"
        reason=">=2 rare heterozygous hits; phase/MOI require manual review"
    elif n_high>=1 or n_plp>=1:
        status="REVIEW"
        reason="single high-impact or clean ClinVar P/LP hit; MOI/second allele review"
    elif n_rare>=1:
        status="CARRIER_OR_MOI_REVIEW"
        reason="single rare coding/splice hit"
    else:
        continue

    gene_rows.append({
        "gene":gene,"status":status,"reason":reason,
        "n_rare_hits":n_rare,"n_het":n_het,"n_hom":n_hom,
        "n_high":n_high,"n_clean_plp":n_plp
    })

gs=pd.DataFrame(gene_rows)
if len(gs):
    order={"HIGH_REVIEW":0,"REVIEW":1,"CARRIER_OR_MOI_REVIEW":2}
    gs["ord"]=gs.status.map(order)
    gs=gs.sort_values(["ord","n_rare_hits"],ascending=[True,False]).drop(columns="ord")

hits["finding_type"]="secondary"
hits.to_parquet(PRIVATE_OUT/"NB04D_secondary_variant_hits_private.parquet",index=False)
gs.to_csv(PRIVATE_OUT/"NB04D_secondary_gene_summary_private.csv",index=False)

display(gs)

## 4. Sanitized public handoff — no secondary gene names

In [ ]:
n_high=int((gs.status=="HIGH_REVIEW").sum()) if len(gs) else 0
n_review=int((gs.status=="REVIEW").sum()) if len(gs) else 0
n_carrier=int((gs.status=="CARRIER_OR_MOI_REVIEW").sum()) if len(gs) else 0

if n_high:
    status="SECONDARY_EXPLANATION_CANDIDATE_REQUIRES_PRIVATE_REVIEW"
elif n_review:
    status="SECONDARY_SINGLE_HIT_REQUIRES_MOI_REVIEW"
else:
    status="NO_STRONG_SECONDARY_CANDIDATE_IN_FROZEN_CODING_SPLICE_PANEL"

handoff={
 "schema":"MVA-NB04D-NEPHROCALCINOSIS-SECONDARY-1.0",
 "created_utc":datetime.now(timezone.utc).isoformat(),
 "status":"COMPLETE",
 "finding_type":"secondary",
 "primary_panel":"NCBI GTR frozen 30-gene nephrolithiasis/nephrocalcinosis panel",
 "n_panel_genes":30,
 "n_filtered_variant_hits":int(len(hits)),
 "n_high_review_genes":n_high,
 "n_review_genes":n_review,
 "n_carrier_or_moi_review_genes":n_carrier,
 "result":status,
 "interpretation":{
   "positive":"private candidate requires phenotype/MOI/segregation/manual clinical review; does not replace BUB1B MVA hypothesis",
   "negative":"no strong coding/splice explanation in frozen panel; does not exclude CNV/SV, deep intronic/regulatory causes, mtDNA, biochemical/acquired causes, or prematurity-associated nephrocalcinosis"
 },
 "privacy":{
   "raw_VCF_reopened":False,
   "full_variant_rows_public":False,
   "secondary_gene_names_public":False,
   "public_handoff_contains_counts_only":True
 }
}
(PUBLIC_OUT/"NB04D_NEPHROCALCINOSIS_HANDOFF.json").write_text(json.dumps(handoff,indent=2))
print(json.dumps(handoff,indent=2))